In [75]:
import arff
import pandas as pd

# загружаем данные
with open("c:/Personal/Science/Data/Binary_classification/telco-customer-churn.arff", encoding="utf-8") as f:
    dataset = arff.load(f)
df = pd.DataFrame(dataset["data"],
                  columns=[a[0] for a in dataset["attributes"]])
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [77]:
from typing import List, Any
import numpy as np

def clean_dataset(
    df: pd.DataFrame,
    cols_to_drop: List[Any] = None,
    missing_values: List[Any] = None
) -> pd.DataFrame:
    """
    Предобработка датафрейма с учетом пользовательских меток пропусков:
    1. Заполняет пропущенные значения
       - Модой для категориальных/текстовых признаков.
       - Средним значением для числовых признаков.
    2. Удаляет признаки, у которых одно значение занимает 100% строк.
    """
    df_cleaned = df.copy()
    n_rows = len(df_cleaned)
    
    if n_rows == 0:
        return df_cleaned

    # Задаем список меток для пропущенных значений по умолчанию, если не передан свой
    if missing_values is None:
        missing_values = ['', ' ', 'N/A', 'NA', 'null', 'nan', 'NaN', 'None', 'none', '?', '-']

    # 1. Приведение пользовательских пропусков к np.nan и их импутация
    for col in df_cleaned.columns:
        # Заменяем все кастомные метки пропусков на np.nan
        df_cleaned[col] = df_cleaned[col].replace(missing_values, np.nan)

        if df_cleaned[col].isna().sum() > 0:
            if pd.api.types.is_numeric_dtype(df_cleaned[col]):
                # а) Числовые переменные -> средним значением
                mean_val = df_cleaned[col].mean()
                df_cleaned[col] = df_cleaned[col].fillna(mean_val)
            else:
                # б) Категориальные переменные -> модой
                mode_val = df_cleaned[col].mode()
                if not mode_val.empty:
                    df_cleaned[col] = df_cleaned[col].fillna(mode_val[0])

    # 2. Проверка условий для удаления колонок
    for col in df_cleaned.columns:
        series = df_cleaned[col]
        max_value_count = series.value_counts().iloc[0]
        max_value_ratio = max_value_count / n_rows

        # Если константных значений 100% -> к удалению
        if max_value_ratio = 1:
            cols_to_drop.append(col)

    # Удаляем выявленные колонки
    if cols_to_drop:
        df_cleaned = df_cleaned.drop(columns=cols_to_drop)
    return df_cleaned

df = clean_dataset(df, ["customerID"])

In [79]:
df.isnull().sum()

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [81]:
df['TotalCharges'] = df['TotalCharges'].astype(float)
df.dtypes

gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object

In [83]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [85]:
df['Churn'] = df['Churn'].map({ 'No' : 0, 'Yes' : 1})
y = df["Churn"]
X = df.drop(["Churn"], axis=1)

In [87]:
import time
import category_encoders as ce
from skrub import SimilarityEncoder, MinHashEncoder
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

from k_fold_encoder import KFoldTargetEncoder
from k_fold_glmm_encoder import KFoldGLMMEncoder

cat_cols = ["gender","MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup", 
            "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies", "Contract", 
            "PaymentMethod", "Partner","Dependents", "PhoneService", "PaperlessBilling"]

encoders = {
   'Leave-one-out': lambda: ce.LeaveOneOutEncoder(cols=cat_cols),
   'K-fold':        lambda: KFoldTargetEncoder(cols=cat_cols),
   "CatBoost":      lambda: ce.CatBoostEncoder(cols=cat_cols),
   "CV-GLMM":       lambda: KFoldGLMMEncoder(cols=cat_cols),
   "WOE":           lambda: ce.WOEEncoder(cols=cat_cols),
   "JS":            lambda: ce.JamesSteinEncoder(cols=cat_cols),
   "MT":            lambda: ce.TargetEncoder(cols=cat_cols),
   'GLMM':          lambda: ce.GLMMEncoder(cols=cat_cols),
   'M-Estimate':    lambda: ce.MEstimateEncoder(cols=cat_cols),
   "OneHot":  lambda: ce.OneHotEncoder(cols=cat_cols),
   "Gray":    lambda: ce.GrayEncoder(cols=cat_cols),
   "BaseN":   lambda: ce.BaseNEncoder(cols=cat_cols),
   "Hashing": lambda: ce.HashingEncoder(cols=cat_cols,n_components=32),
   "Helmert": lambda: ce.HelmertEncoder(cols=cat_cols),
   "Sum":     lambda: ce.SumEncoder(cols=cat_cols),
   "BackwardDifference": lambda: ce.BackwardDifferenceEncoder(cols=cat_cols),
   "MinHash": lambda: ColumnTransformer([(
                f"minhash_{col}",
                MinHashEncoder(),
                [col]
            )
            for col in cat_cols
        ],
        remainder="passthrough"
    ),
    "Similarity": lambda: ColumnTransformer([
        ("cat", SimilarityEncoder(), cat_cols)], 
        remainder="passthrough")
}

models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), activation="identity", max_iter=1000, random_state=42),
    "LinearSVM": LinearSVC(dual="auto", random_state=42),
    "CART": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(random_state=42, n_jobs=4),
    "XGBoost": XGBClassifier(eval_metric="logloss", n_jobs=4),
    "LightGBM": LGBMClassifier(verbose=-1, n_jobs=4),
    "CatBoost": CatBoostClassifier(verbose=0, thread_count=4)
}

records = []

# Инициализируем генератор разбиений
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

for model_name, model_template in models.items():
    for encoder_name, encoder in encoders.items():
        
        split_idx = 0
        # Проходим по всем фолдам вручную
        for train_idx, test_idx in cv.split(X, y):
            start_time = time.time()

            # 1. Разбиение выборки на train и test
            X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # 2. Инициализация и обработка кодировщиком
            enc_instance = encoder()  # Вызываем lambda, получая объект кодировщика
            X_train_encoded = enc_instance.fit_transform(X_train, y_train)
            X_test_encoded = enc_instance.transform(X_test)

            # 3. Масштабирование (если требуется для модели)
            if model_name in ["LogReg", "LinearSVM", "MLP"]:
                scaler = StandardScaler()
                X_train_processed = scaler.fit_transform(X_train_encoded)
                X_test_processed = scaler.transform(X_test_encoded)
            else:
                X_train_processed = X_train_encoded
                X_test_processed = X_test_encoded

            # 4. Обучение модели (клонируем или создаем заново, чтобы сбросить состояние)
            model = clone(model_template)
            model.fit(X_train_processed, y_train)

            # 5. Получение предсказаний
            y_pred = model.predict(X_test_processed)

            # Для ROC AUC и PR AUC нужны вероятности или decision_function
            if hasattr(model, "predict_proba"):
                y_proba = model.predict_proba(X_test_processed)[:, 1]
            elif hasattr(model, "decision_function"):
                y_proba = model.decision_function(X_test_processed)
            else:
                y_proba = y_pred

            # 6. Расчет метрик качество (scoring)
            auc_score = roc_auc_score(y_test, y_proba)
            pr_auc_score = average_precision_score(y_test, y_proba)
            acc_score = accuracy_score(y_test, y_pred)
            f1_val = f1_score(y_test, y_pred)
            prec_val = precision_score(y_test, y_pred)
            rec_val = recall_score(y_test, y_pred)

            elapsed_time = time.time() - start_time

            # 7. Вычисление номера повтора (repeat) и фолда (fold)
            repeat = split_idx // 5 + 1
            fold = split_idx % 5 + 1

            records.append(
                {
                    "approach": "Empirical",
                    "dataset": "TelcoChurn",
                    "model": model_name,
                    "encoder": encoder_name,
                    "repeat": repeat,
                    "fold": fold,
                    "auc": auc_score,
                    "pr_auc": pr_auc_score,
                    "accuracy": acc_score,
                    "f1": f1_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "total_time ": elapsed_time,
                }
            )

            print(
                f"Empirical, TelcoChurn, {model_name}, {encoder_name}, {repeat}, {fold}, "
                f"{auc_score}, {pr_auc_score}, {acc_score}, {f1_val}, "
                f"{prec_val}, {rec_val}, {elapsed_time}"
            )

            split_idx += 1
results = pd.DataFrame(records)
results.to_csv("results_telco_churn_empirical.csv", index=True)
print("Done!")

Empirical, TelcoChurn, LogReg, Leave-one-out, 1, 1, 0.8557506006355111, 0.6597173852454261, 0.808374733853797, 0.6142857142857143, 0.6595092024539877, 0.5748663101604278, 0.19499707221984863
Empirical, TelcoChurn, LogReg, Leave-one-out, 1, 2, 0.801348523599163, 0.5553405502539157, 0.7806955287437899, 0.5553956834532374, 0.6012461059190031, 0.516042780748663, 0.16395950317382812
Empirical, TelcoChurn, LogReg, Leave-one-out, 1, 3, 0.8615179932315482, 0.6981047126537503, 0.8183108587650816, 0.6311239193083573, 0.684375, 0.5855614973262032, 0.16075515747070312
Empirical, TelcoChurn, LogReg, Leave-one-out, 1, 4, 0.8243501573609977, 0.6156031966385695, 0.7876420454545454, 0.5543964232488823, 0.6241610738255033, 0.49865951742627346, 0.17112231254577637
Empirical, TelcoChurn, LogReg, Leave-one-out, 1, 5, 0.8215525088178405, 0.6090894654670698, 0.7784090909090909, 0.5343283582089552, 0.6047297297297297, 0.4786096256684492, 0.16991019248962402
Empirical, TelcoChurn, LogReg, Leave-one-out, 2, 1, 

C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, TelcoChurn, CART, Leave-one-out, 1, 1, 0.5, 0.2654364797728886, 0.7345635202271115, 0.0, 0.0, 0.0, 0.2474048137664795


C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, TelcoChurn, CART, Leave-one-out, 1, 2, 0.5, 0.2654364797728886, 0.7345635202271115, 0.0, 0.0, 0.0, 0.23572826385498047
Empirical, TelcoChurn, CART, Leave-one-out, 1, 3, 0.5, 0.2654364797728886, 0.7345635202271115, 0.0, 0.0, 0.0, 0.19622397422790527


C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


... (197 outputs left out)

C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, TelcoChurn, CatBoost, Leave-one-out, 5, 4, 0.4758285736488324, 0.24853660934272948, 0.7350852272727273, 0.0, 0.0, 0.0, 2.184495449066162


C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, TelcoChurn, CatBoost, Leave-one-out, 5, 5, 0.46984608860248867, 0.2502103001714072, 0.734375, 0.0, 0.0, 0.0, 2.1695945262908936
Empirical, TelcoChurn, CatBoost, K-fold, 1, 1, 0.8611924875351986, 0.6773863836167239, 0.8105039034776437, 0.613603473227207, 0.668769716088328, 0.5668449197860963, 2.6281845569610596
Empirical, TelcoChurn, CatBoost, K-fold, 1, 2, 0.8426632049394198, 0.6565107975154583, 0.7998580553584103, 0.5994318181818182, 0.6393939393939394, 0.5641711229946524, 2.5961952209472656
Empirical, TelcoChurn, CatBoost, K-fold, 1, 3, 0.854062362758015, 0.675477133701217, 0.8055358410220014, 0.5994152046783626, 0.6612903225806451, 0.5481283422459893, 2.7044894695281982
Empirical, TelcoChurn, CatBoost, K-fold, 1, 4, 0.8286565385761097, 0.6300899422908512, 0.7855113636363636, 0.5396341463414634, 0.6254416961130742, 0.4745308310991957, 2.7395734786987305
Empirical, TelcoChurn, CatBoost, K-fold, 1, 5, 0.838087899130111, 0.6578948093921712, 0.8039772727272727, 0.5766871165644